# Lasso Regression — Built from Scratch

This notebook implements **Lasso Regression** entirely from scratch using only NumPy,
then evaluates it on the pre-processed student performance dataset.

Lasso adds an L1 penalty to ordinary least squares:

$$\hat{\beta} = \arg\min_\beta \; \|y - X\beta\|_2^2 + \lambda\|\beta\|_1$$

Unlike Ridge, **L1 produces exact zeros** — Lasso performs built-in feature selection.
There is no closed-form solution, so we use **Coordinate Descent**:
cycle through each coefficient and update it while holding all others fixed.

| Task | Target | Metric |
|------|--------|--------|
| Regression | `exam_score` | RMSE, MAE, R² |
| Classification | `dropout_risk` | Accuracy, F1, ROC-AUC |

Mirrors the structure of `decision_trees.ipynb` and `ridge_regression.ipynb` for easy comparison.

---
## 1 · Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings

from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = './processed'
OUT_DIR       = '../project/outputs/lasso_regression'
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/models',  exist_ok=True)

print('Config OK. Output directory:', OUT_DIR)

---
## 2 · Load Pre-Processed Artefacts

In [ ]:
# Scaled splits — linear models require standardised features
X_train = pd.read_csv(f'{PROCESSED_DIR}/X_train_scaled.csv', index_col=0)
X_val   = pd.read_csv(f'{PROCESSED_DIR}/X_val_scaled.csv',   index_col=0)
X_test  = pd.read_csv(f'{PROCESSED_DIR}/X_test_scaled.csv',  index_col=0)

# Regression targets
y_reg_train = pd.read_csv(f'{PROCESSED_DIR}/y_reg_train.csv', index_col=0).squeeze()
y_reg_val   = pd.read_csv(f'{PROCESSED_DIR}/y_reg_val.csv',   index_col=0).squeeze()
y_reg_test  = pd.read_csv(f'{PROCESSED_DIR}/y_reg_test.csv',  index_col=0).squeeze()

# Classification targets
y_clf_train = pd.read_csv(f'{PROCESSED_DIR}/y_clf_train.csv', index_col=0).squeeze()
y_clf_val   = pd.read_csv(f'{PROCESSED_DIR}/y_clf_val.csv',   index_col=0).squeeze()
y_clf_test  = pd.read_csv(f'{PROCESSED_DIR}/y_clf_test.csv',  index_col=0).squeeze()

feature_names = pd.read_csv(f'{PROCESSED_DIR}/feature_list.csv')['feature'].tolist()

print('Loaded splits:')
for name, arr in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    print(f'  {name:5s}  {arr.shape[0]:6d} rows  {arr.shape[1]} features')
print(f'\nRegression target  — mean={y_reg_train.mean():.2f}, std={y_reg_train.std():.2f}')
print(f'Classification target — class balance {y_clf_train.value_counts(normalize=True).round(3).to_dict()}')

---
## 3 · Lasso Regression — Built from Scratch

### 3.1 · The Maths — Coordinate Descent

The L1 penalty is not differentiable at zero, so gradient descent cannot be applied directly.
**Coordinate Descent** sidesteps this by solving a 1-D sub-problem per coefficient.

For each coefficient $j$, compute the **partial residual** (residual without feature $j$'s contribution):
$$r_j = y - \sum_{k \neq j} x_k \beta_k$$

The optimal $\beta_j$ update is then given by the **soft-thresholding operator** $S$:
$$\beta_j \leftarrow S\!\left(\frac{x_j^T r_j}{x_j^T x_j},\; \frac{\lambda}{x_j^T x_j}\right)$$

where:
$$S(z, \gamma) = \text{sign}(z)\max(|z| - \gamma,\; 0)$$

This sets $\beta_j = 0$ whenever $|z| \leq \gamma$ — producing exact sparsity.

In [ ]:
class LassoRegressionScratch:
    """
    Lasso Regression via Coordinate Descent, implemented from scratch.

    The intercept is fit separately (not penalised) by centring y.

    Parameters
    ----------
    lambda_  : L1 regularisation strength
    max_iter : max coordinate descent cycles
    tol      : convergence tolerance on max coefficient change
    """

    def __init__(self, lambda_=1.0, max_iter=1000, tol=1e-6):
        self.lambda_  = lambda_
        self.max_iter = max_iter
        self.tol      = tol
        self.coef_    = None
        self.intercept_ = 0.0
        self.n_iter_    = 0

    @staticmethod
    def _soft_threshold(z, gamma):
        """Soft-thresholding operator — the Lasso update step."""
        return np.sign(z) * np.maximum(np.abs(z) - gamma, 0.0)

    def fit(self, X, y):
        X_ = np.asarray(X, dtype=float)
        y_ = np.asarray(y, dtype=float)
        n, p = X_.shape

        # Centre y and X (intercept handled via means, not penalised)
        self.intercept_ = y_.mean()
        y_c = y_ - self.intercept_

        # Pre-compute column norms squared (constant across iterations)
        col_norms_sq = (X_ ** 2).sum(axis=0)   # shape: (p,)
        # Guard against zero-variance columns
        col_norms_sq = np.where(col_norms_sq == 0, 1.0, col_norms_sq)

        # Initialise coefficients at zero
        self.coef_ = np.zeros(p)

        for iteration in range(self.max_iter):
            beta_old = self.coef_.copy()

            for j in range(p):
                # Partial residual: y - X*beta + x_j * beta_j
                r_j = y_c - X_ @ self.coef_ + X_[:, j] * self.coef_[j]

                # OLS update for beta_j
                z_j = X_[:, j] @ r_j / col_norms_sq[j]

                # Lasso soft-threshold
                gamma = self.lambda_ / col_norms_sq[j]
                self.coef_[j] = self._soft_threshold(z_j, gamma)

            # Update intercept to account for current coefficients
            self.intercept_ = y_.mean() - X_.mean(axis=0) @ self.coef_

            # Convergence check
            if np.max(np.abs(self.coef_ - beta_old)) < self.tol:
                self.n_iter_ = iteration + 1
                break
        else:
            self.n_iter_ = self.max_iter

        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    @property
    def n_nonzero_(self):
        return np.sum(self.coef_ != 0)


# ── Sanity check against sklearn ────────────────────────────────────────────
from sklearn.linear_model import Lasso as SklearnLasso

test_lambda = 0.01
scratch = LassoRegressionScratch(lambda_=test_lambda * len(X_train), max_iter=2000).fit(X_train, y_reg_train)
sk      = SklearnLasso(alpha=test_lambda, max_iter=5000).fit(X_train, y_reg_train)

scratch_rmse = np.sqrt(mean_squared_error(y_reg_val, scratch.predict(X_val)))
sk_rmse      = np.sqrt(mean_squared_error(y_reg_val, sk.predict(X_val)))

print(f'Scratch  val RMSE : {scratch_rmse:.6f}  (non-zero coefs: {scratch.n_nonzero_})')
print(f'Sklearn  val RMSE : {sk_rmse:.6f}  (non-zero coefs: {(sk.coef_ != 0).sum()})')
print(f'Iterations        : {scratch.n_iter_}')

---
## 4 · Regression Task — Predict `exam_score`

### 4.1 · λ vs RMSE Curve (Validation Set)

Mirrors the *Depth vs RMSE* curve from the decision tree notebook.

In [ ]:
# Note: our lambda_ absorbs the n factor, so we search in raw λ/n units here.
n_train = len(X_train)
alphas  = np.logspace(-4, 2, 50)   # sklearn-style alpha values

train_rmses, val_rmses, n_nonzeros = [], [], []

for alpha in alphas:
    m = LassoRegressionScratch(lambda_=alpha * n_train, max_iter=2000)
    m.fit(X_train, y_reg_train)
    train_rmses.append(np.sqrt(mean_squared_error(y_reg_train, m.predict(X_train))))
    val_rmses.append(np.sqrt(mean_squared_error(y_reg_val,     m.predict(X_val))))
    n_nonzeros.append(m.n_nonzero_)

best_alpha_idx = np.argmin(val_rmses)
best_alpha_reg = alphas[best_alpha_idx]

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.semilogx(alphas, train_rmses, 'o-', markersize=3, label='Train RMSE', color='steelblue')
ax1.semilogx(alphas, val_rmses,   's-', markersize=3, label='Val RMSE',   color='coral')
ax1.axvline(best_alpha_reg, color='red', linestyle='--',
            label=f'Best α = {best_alpha_reg:.5f}')
ax1.set_xlabel('α (log scale)')
ax1.set_ylabel('RMSE')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.semilogx(alphas, n_nonzeros, '--', color='gray', alpha=0.5, label='# non-zero coefs')
ax2.set_ylabel('Non-zero coefficients', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.legend(loc='upper right')

plt.title('Lasso Regression — α vs RMSE & Sparsity')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_alpha_vs_rmse.png', dpi=150)
plt.show()
print(f'Best α: {best_alpha_reg:.5f}  val RMSE={val_rmses[best_alpha_idx]:.4f}  non-zero={n_nonzeros[best_alpha_idx]}')

### 4.2 · Coefficient Regularisation Path (Lasso)

Unlike Ridge, coefficients hit **exactly zero** as α increases — visible as lines vanishing.

In [ ]:
path_alphas = np.logspace(-4, 1, 60)
coef_paths  = []

for alpha in path_alphas:
    m = LassoRegressionScratch(lambda_=alpha * n_train, max_iter=2000)
    m.fit(X_train, y_reg_train)
    coef_paths.append(m.coef_.copy())

coef_paths = np.array(coef_paths)   # (n_alphas, n_features)

# Identify top-10 features by absolute coefficient at the best alpha
best_model_path = LassoRegressionScratch(lambda_=best_alpha_reg * n_train, max_iter=2000)
best_model_path.fit(X_train, y_reg_train)
top10_idx = np.argsort(np.abs(best_model_path.coef_))[::-1][:10]

plt.figure(figsize=(11, 5))
for i in range(len(feature_names)):
    if i in top10_idx:
        plt.semilogx(path_alphas, coef_paths[:, i],
                     lw=1.8, label=feature_names[i])
    else:
        plt.semilogx(path_alphas, coef_paths[:, i],
                     color='lightgray', lw=0.6, alpha=0.5)
plt.axvline(best_alpha_reg, color='red', linestyle='--', lw=1.2, label=f'Best α={best_alpha_reg:.4f}')
plt.axhline(0, color='black', lw=0.8)
plt.xlabel('α (log scale)')
plt.ylabel('Coefficient value')
plt.title('Lasso Regularisation Path — Top 10 Features (notice coefficients hitting zero)')
plt.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_regularisation_path.png', dpi=150)
plt.show()

### 4.3 · Fit Best Model on Train + Val, Evaluate on Test

In [ ]:
X_trainval     = pd.concat([X_train, X_val])
y_reg_trainval = pd.concat([y_reg_train, y_reg_val])
n_trainval     = len(X_trainval)

best_lasso_reg = LassoRegressionScratch(lambda_=best_alpha_reg * n_trainval, max_iter=3000)
best_lasso_reg.fit(X_trainval, y_reg_trainval)

y_reg_pred = best_lasso_reg.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
mae  = mean_absolute_error(y_reg_test, y_reg_pred)
r2   = r2_score(y_reg_test, y_reg_pred)

print('─' * 40)
print(f'  Best α       : {best_alpha_reg:.5f}')
print(f'  Non-zero coef: {best_lasso_reg.n_nonzero_} / {X_train.shape[1]}')
print(f'  RMSE         : {rmse:.3f}')
print(f'  MAE          : {mae:.3f}')
print(f'  R²           : {r2:.4f}')
print('─' * 40)

### 4.4 · Actual vs Predicted Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.3, s=8, color='steelblue')
lims = [y_reg_test.min(), y_reg_test.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual exam_score')
axes[0].set_ylabel('Predicted exam_score')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})')
axes[0].legend()

residuals = y_reg_test.values - y_reg_pred
axes[1].scatter(y_reg_pred, residuals, alpha=0.3, s=8, color='coral')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted exam_score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle('Lasso Regression — Test Set Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_actual_vs_predicted.png', dpi=150)
plt.show()

### 4.5 · Feature Importances (Coefficient Magnitudes)

Lasso's sparsity makes this plot especially informative: many coefficients will be exactly zero.

In [ ]:
importances_reg = pd.Series(np.abs(best_lasso_reg.coef_), index=feature_names)
top20_reg       = importances_reg.sort_values(ascending=False).head(20)
n_zero          = (best_lasso_reg.coef_ == 0).sum()

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'lightgray' for v in top20_reg]
top20_reg.plot(kind='barh', color=colors)
plt.gca().invert_yaxis()
plt.xlabel('|Coefficient| (standardised features)')
plt.title(f'Lasso Regression — Top 20 Feature Importances\n'
          f'({best_lasso_reg.n_nonzero_} non-zero, {n_zero} zeroed out of {X_train.shape[1]})')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_feature_importances.png', dpi=150)
plt.show()

### 4.6 · Sparsity Profile — All Coefficients

In [ ]:
all_coefs = pd.Series(best_lasso_reg.coef_, index=feature_names).sort_values()

colors = ['coral' if v < 0 else ('steelblue' if v > 0 else 'lightgray')
          for v in all_coefs]

plt.figure(figsize=(14, max(5, len(feature_names) * 0.22)))
plt.barh(all_coefs.index, all_coefs.values, color=colors)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Coefficient value (standardised features)')
plt.title('Lasso Regression — Full Coefficient Profile\n(grey = exactly zero; Lasso feature selection)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_full_coef_profile.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 · Residual Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(0, color='red', linestyle='--', lw=1.5)
plt.xlabel('Residual (Actual − Predicted)')
plt.ylabel('Count')
plt.title('Lasso Regression — Residual Distribution')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/reg_residual_distribution.png', dpi=150)
plt.show()
print(f'Residual mean : {residuals.mean():.4f}')
print(f'Residual std  : {residuals.std():.4f}')

### 4.8 · Save Regression Model

In [ ]:
joblib.dump(best_lasso_reg, f'{OUT_DIR}/models/lasso_reg.joblib')
print(f'Lasso regression model saved  (α={best_alpha_reg:.5f}, non-zero={best_lasso_reg.n_nonzero_})')

---
## 5 · Classification Task — Predict `dropout_risk`

We implement **Lasso-regularised Logistic Regression** from scratch using
**proximal gradient descent** — the standard approach for L1-penalised smooth losses.

Each iteration:
1. **Gradient step** on the logistic loss (smooth part)
2. **Proximal step** via soft-thresholding on the L1 term (non-smooth part)

$$\beta \leftarrow S\!\left(\beta - \eta \nabla_{\text{BCE}}(\beta),\; \eta\lambda\right)$$

In [ ]:
class LassoLogisticScratch:
    """
    Logistic Regression with L1 (Lasso) regularisation via proximal
    gradient descent (ISTA), implemented from scratch.

    Parameters
    ----------
    lambda_    : L1 penalty strength
    lr         : gradient step size (learning rate)
    epochs     : max iterations
    batch_size : mini-batch size (None = full batch)
    patience   : early-stopping patience on val loss
    """

    def __init__(self, lambda_=0.01, lr=0.05, epochs=500,
                 batch_size=1024, patience=20):
        self.lambda_    = lambda_
        self.lr         = lr
        self.epochs     = epochs
        self.batch_size = batch_size
        self.patience   = patience
        self.beta_      = None
        self.history_   = {'train_loss': [], 'val_loss': []}

    @staticmethod
    def _add_bias(X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    @staticmethod
    def _sigmoid(z):
        return np.where(z >= 0,
                        1 / (1 + np.exp(-z)),
                        np.exp(z) / (1 + np.exp(z)))

    @staticmethod
    def _soft_threshold(beta, threshold):
        """Apply soft-threshold to all except intercept (index 0)."""
        out    = np.sign(beta) * np.maximum(np.abs(beta) - threshold, 0.0)
        out[0] = beta[0]   # never penalise intercept
        return out

    def _bce_loss(self, X_b, y, beta):
        p   = np.clip(self._sigmoid(X_b @ beta), 1e-12, 1 - 1e-12)
        bce = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        reg = self.lambda_ * np.sum(np.abs(beta[1:]))
        return bce + reg

    def fit(self, X, y, X_val=None, y_val=None):
        X_b  = self._add_bias(np.asarray(X, dtype=float))
        y_   = np.asarray(y, dtype=float)
        n, p = X_b.shape

        rng        = np.random.default_rng(42)
        self.beta_ = rng.normal(0, 0.01, size=p)

        best_val    = float('inf')
        best_beta   = self.beta_.copy()
        patience_ct = 0

        X_val_b = self._add_bias(np.asarray(X_val, dtype=float)) if X_val is not None else None
        y_val_  = np.asarray(y_val, dtype=float) if y_val is not None else None

        bs = self.batch_size if self.batch_size else n

        for epoch in range(1, self.epochs + 1):
            idx    = rng.permutation(n)
            X_shuf = X_b[idx]
            y_shuf = y_[idx]

            for start in range(0, n, bs):
                Xb    = X_shuf[start:start + bs]
                yb    = y_shuf[start:start + bs]
                # 1. Gradient of BCE loss
                p_hat = self._sigmoid(Xb @ self.beta_)
                grad  = (Xb.T @ (p_hat - yb)) / len(yb)
                # 2. Gradient step
                beta_half = self.beta_ - self.lr * grad
                # 3. Proximal (soft-threshold) step for L1
                self.beta_ = self._soft_threshold(beta_half, self.lr * self.lambda_)

            train_loss = self._bce_loss(X_b, y_, self.beta_)
            self.history_['train_loss'].append(train_loss)

            if X_val_b is not None:
                val_loss = self._bce_loss(X_val_b, y_val_, self.beta_)
                self.history_['val_loss'].append(val_loss)

                if val_loss < best_val:
                    best_val    = val_loss
                    best_beta   = self.beta_.copy()
                    patience_ct = 0
                else:
                    patience_ct += 1
                    if patience_ct >= self.patience:
                        print(f'  Early stop at epoch {epoch}  (best val loss={best_val:.5f})')
                        break

            if epoch % 50 == 0:
                vl = f"{self.history_['val_loss'][-1]:.5f}" if self.history_['val_loss'] else 'N/A'
                nz = (self.beta_[1:] != 0).sum()
                print(f'  Epoch {epoch:4d} | train={train_loss:.5f} | val={vl} | non-zero={nz}')

        if X_val_b is not None:
            self.beta_ = best_beta
        return self

    def predict_proba(self, X):
        return self._sigmoid(self._add_bias(np.asarray(X, dtype=float)) @ self.beta_)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    @property
    def intercept_(self):
        return self.beta_[0]

    @property
    def coef_(self):
        return self.beta_[1:]

    @property
    def n_nonzero_(self):
        return (self.coef_ != 0).sum()


print('LassoLogisticScratch class defined.')

### 5.1 · λ vs Val AUC & Sparsity

In [ ]:
clf_lambdas = np.logspace(-4, 0, 16)
val_aucs, val_nz = [], []

for lam in clf_lambdas:
    m = LassoLogisticScratch(lambda_=lam, lr=0.05, epochs=300, patience=15)
    m.fit(X_train, y_clf_train, X_val, y_clf_val)
    val_aucs.append(roc_auc_score(y_clf_val, m.predict_proba(X_val)))
    val_nz.append(m.n_nonzero_)
    print(f'  λ={lam:.5f}  val AUC={val_aucs[-1]:.4f}  non-zero={val_nz[-1]}')

best_lam_clf_idx = np.argmax(val_aucs)
best_lam_clf     = clf_lambdas[best_lam_clf_idx]

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.semilogx(clf_lambdas, val_aucs, 's-', color='mediumseagreen', label='Val AUC')
ax1.axvline(best_lam_clf, color='red', linestyle='--', label=f'Best λ={best_lam_clf:.5f}')
ax1.set_xlabel('λ (log scale)')
ax1.set_ylabel('Val ROC-AUC')
ax1.legend(loc='upper right')

ax2 = ax1.twinx()
ax2.semilogx(clf_lambdas, val_nz, '--', color='gray', alpha=0.5, label='# non-zero coefs')
ax2.set_ylabel('Non-zero coefficients', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')
ax2.legend(loc='center right')

plt.title('Lasso Logistic — λ vs Val AUC & Sparsity')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_lambda_vs_auc.png', dpi=150)
plt.show()
print(f'Best λ: {best_lam_clf:.5f}  (val AUC={val_aucs[best_lam_clf_idx]:.4f}, non-zero={val_nz[best_lam_clf_idx]})')

### 5.2 · Train Best Classifier & Show Loss Curve

In [ ]:
best_lasso_clf = LassoLogisticScratch(lambda_=best_lam_clf, lr=0.05, epochs=500, patience=20)
best_lasso_clf.fit(X_train, y_clf_train, X_val, y_clf_val)

plt.figure(figsize=(9, 4))
plt.plot(best_lasso_clf.history_['train_loss'], label='Train loss')
plt.plot(best_lasso_clf.history_['val_loss'],   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('L1-regularised Binary Cross-Entropy')
plt.title(f'Lasso Logistic — Training Loss Curve  (λ={best_lam_clf:.5f})')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_loss_curve.png', dpi=150)
plt.show()

### 5.3 · Evaluate on Test Set

In [ ]:
y_proba = best_lasso_clf.predict_proba(X_test)
y_pred  = best_lasso_clf.predict(X_test)

acc = accuracy_score(y_clf_test, y_pred)
f1  = f1_score(y_clf_test, y_pred)
auc = roc_auc_score(y_clf_test, y_proba)

print('─' * 40)
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print(f'  Non-zero  : {best_lasso_clf.n_nonzero_} / {X_train.shape[1]}')
print('─' * 40)
print()
print(classification_report(y_clf_test, y_pred, target_names=['No Risk', 'Dropout Risk']))

### 5.4 · Confusion Matrix

In [ ]:
cm = confusion_matrix(y_clf_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Risk', 'Dropout Risk'],
            yticklabels=['No Risk', 'Dropout Risk'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Lasso Logistic — Confusion Matrix\n(Accuracy={acc:.3f}, F1={f1:.3f})')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_confusion_matrix.png', dpi=150)
plt.show()

### 5.5 · ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_clf_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='mediumseagreen', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Lasso Logistic — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_roc_curve.png', dpi=150)
plt.show()

### 5.6 · Feature Importances & Sparsity Profile

In [ ]:
importances_clf = pd.Series(np.abs(best_lasso_clf.coef_), index=feature_names)
top20_clf       = importances_clf.sort_values(ascending=False).head(20)
n_zero_clf      = (best_lasso_clf.coef_ == 0).sum()

plt.figure(figsize=(10, 6))
colors = ['mediumseagreen' if v > 0 else 'lightgray' for v in top20_clf]
top20_clf.plot(kind='barh', color=colors)
plt.gca().invert_yaxis()
plt.xlabel('|Coefficient| (standardised features)')
plt.title(f'Lasso Logistic — Top 20 Feature Importances\n'
          f'({best_lasso_clf.n_nonzero_} non-zero, {n_zero_clf} zeroed out of {X_train.shape[1]})')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_feature_importances.png', dpi=150)
plt.show()

### 5.7 · Full Coefficient Profile — All Features

In [ ]:
all_coefs_clf = pd.Series(best_lasso_clf.coef_, index=feature_names).sort_values()
colors_clf    = ['coral' if v < 0 else ('mediumseagreen' if v > 0 else 'lightgray')
                 for v in all_coefs_clf]

plt.figure(figsize=(14, max(5, len(feature_names) * 0.22)))
plt.barh(all_coefs_clf.index, all_coefs_clf.values, color=colors_clf)
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Coefficient value (standardised features)')
plt.title('Lasso Logistic — Full Coefficient Profile\n(grey = exactly zero; green = positive; red = negative)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/figures/clf_full_coef_profile.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.8 · Save Classification Model

In [ ]:
joblib.dump(best_lasso_clf, f'{OUT_DIR}/models/lasso_clf.joblib')
print(f'Lasso logistic model saved  (λ={best_lam_clf:.5f}, non-zero={best_lasso_clf.n_nonzero_})')

---
## 6 · Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        'Task':         'Regression (exam_score)',
        'Best α':       f'{best_alpha_reg:.5f}',
        'Non-zero':     f'{best_lasso_reg.n_nonzero_} / {X_train.shape[1]}',
        'Test RMSE':    f'{rmse:.3f}',
        'Test MAE':     f'{mae:.3f}',
        'Test R²':      f'{r2:.4f}',
        'Test Acc':     '—',
        'Test F1':      '—',
        'Test AUC':     '—',
    },
    {
        'Task':         'Classification (dropout_risk)',
        'Best α':       f'{best_lam_clf:.5f}',
        'Non-zero':     f'{best_lasso_clf.n_nonzero_} / {X_train.shape[1]}',
        'Test RMSE':    '—',
        'Test MAE':     '—',
        'Test R²':      '—',
        'Test Acc':     f'{acc:.4f}',
        'Test F1':      f'{f1:.4f}',
        'Test AUC':     f'{auc:.4f}',
    },
])
display(summary)

print('\nAll figures saved to:', f'{OUT_DIR}/figures/')
print('All models saved to: ', f'{OUT_DIR}/models/')